# Практическая работа 7. Задание 1
A* на взвешенном графе, манхэттенская эвристика по координатам рисунка

In [1]:
import heapq

graph = {
    'A': [('B', 3), ('C', 5)],
    'B': [('A', 3), ('E', 7), ('D', 4)],
    'C': [('A', 5), ('D', 2), ('F', 6)],
    'D': [('B', 4), ('C', 2), ('E', 3), ('G', 8)],
    'E': [('B', 7), ('D', 3), ('H', 5)],
    'F': [('C', 6), ('G', 4)],
    'G': [('D', 8), ('F', 4), ('H', 2), ('T', 7)],
    'H': [('E', 5), ('G', 2), ('T', 6)],
    'T': [('G', 7), ('H', 6)]
}
# координаты вершин по рисунку (условные клетки)
coords = {'A': (0, 0), 'B': (1, 2), 'C': (1, -2), 'D': (2, 0), 'E': (2, 3),
          'F': (2, -3), 'G': (3, -1), 'H': (3, 3), 'T': (4, 1)}


def h(v):
    # манхэттенское расстояние от v до T
    x1, y1 = coords[v]
    x2, y2 = coords['T']
    return abs(x1 - x2) + abs(y1 - y2)


def astar(graph, start, goal, h):
    open_set = [(h(start), start)]
    g_score = {start: 0}
    came_from = {}
    while open_set:
        _, current = heapq.heappop(open_set)
        if current == goal:
            path = [current]
            while current in came_from:
                current = came_from[current]
                path.append(current)
            return path[::-1], g_score[goal]
        for neighbor, weight in graph[current]:
            tentative_g = g_score[current] + weight
            if neighbor not in g_score or tentative_g < g_score[neighbor]:
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g
                f = tentative_g + h(neighbor)
                heapq.heappush(open_set, (f, neighbor))
    return None, float('inf')


def dijkstra_all(graph, start):
    # расстояния от start до всех вершин
    dist = {v: float('inf') for v in graph}
    dist[start] = 0
    pq = [(0, start)]
    while pq:
        d, v = heapq.heappop(pq)
        if d > dist[v]:
            continue
        for nb, w in graph[v]:
            if d + w < dist[nb]:
                dist[nb] = d + w
                heapq.heappush(pq, (dist[nb], nb))
    return dist

In [2]:
# допустимость: h(v) не больше настоящего расстояния до T
real = dijkstra_all(graph, 'T')
print('Вершина   h   d(v,T)')
for v in graph:
    ok = 'ok' if h(v) <= real[v] else 'переоценка'
    print('   %s     %2d     %2d   %s' % (v, h(v), real[v], ok))

path, cost = astar(graph, 'A', 'T', h)
print()
print('A*:       путь', ' -> '.join(path), ', стоимость', cost)

# Дейкстра = A* с h = 0
path_d, cost_d = astar(graph, 'A', 'T', lambda v: 0)
print('Дейкстра: путь', ' -> '.join(path_d), ', стоимость', cost_d)

Вершина   h   d(v,T)
   A      5     21   ok
   B      4     18   ok
   C      6     16   ok
   D      3     14   ok
   E      4     11   ok
   F      6     11   ok
   G      3      7   ok
   H      3      6   ok
   T      0      0   ok

A*:       путь A -> B -> E -> H -> T , стоимость 21
Дейкстра: путь A -> B -> E -> H -> T , стоимость 21
